# Domain-Specific Fine-Tuning Demo
## Challenge 1: Software Domain EN→NL Translation

This notebook demonstrates the complete fine-tuning pipeline for domain-specific machine translation.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import torch
import pandas as pd
from transformers import MarianMTModel, MarianTokenizer
from peft import PeftModel

from config import get_config
from data_loader import DataLoaderFactory
from evaluation import TranslationEvaluator

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 2. Load Test Dataset

In [ ]:
# Load software domain test set
config = get_config()
data_factory = DataLoaderFactory(config)

sources, references = data_factory.load_software_test_set()

print(f"Loaded {len(sources)} test samples")
print("\nSample pairs:")
for i in range(5):
    print(f"\n[{i+1}] EN: {sources[i]}")
    print(f"    NL: {references[i]}")

## 3. Baseline Model Evaluation

In [ ]:
# Load baseline MarianMT model
model_name = "Helsinki-NLP/opus-mt-en-nl"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)
model.eval()

print(f"Model loaded on {device}")

In [ ]:
def translate_batch(texts, model, tokenizer, device, batch_size=16):
    """Translate a list of texts"""
    translations = []
    
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128
            ).to(device)
            
            outputs = model.generate(**inputs, max_length=128, num_beams=4)
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            translations.extend(decoded)
    
    return translations

# Translate test set
baseline_translations = translate_batch(sources, model, tokenizer, device)

print("Sample translations:")
for i in range(5):
    print(f"\n[{i+1}]")
    print(f"Source:     {sources[i]}")
    print(f"Baseline:   {baseline_translations[i]}")
    print(f"Reference:  {references[i]}")

## 4. Compute Evaluation Metrics

In [ ]:
import sacrebleu
from sacrebleu.metrics import BLEU, CHRF, TER

# Initialize metrics
bleu = BLEU()
chrf = CHRF()
ter = TER()

# Compute scores
refs = [[ref] for ref in references]

bleu_result = bleu.corpus_score(baseline_translations, refs)
chrf_result = chrf.corpus_score(baseline_translations, refs)
ter_result = ter.corpus_score(baseline_translations, refs)

print("=" * 50)
print("BASELINE MODEL EVALUATION (Software Domain)")
print("=" * 50)
print(f"BLEU:  {bleu_result.score:.2f}")
print(f"  - BLEU-1: {bleu_result.precisions[0]:.2f}")
print(f"  - BLEU-2: {bleu_result.precisions[1]:.2f}")
print(f"  - BLEU-3: {bleu_result.precisions[2]:.2f}")
print(f"  - BLEU-4: {bleu_result.precisions[3]:.2f}")
print(f"chrF:  {chrf_result.score:.2f}")
print(f"TER:   {ter_result.score:.2f}")
print("=" * 50)

## 5. Encoder-Decoder Fine-Tuning (Quick Demo)

In [ ]:
# This cell demonstrates the fine-tuning setup
# For full training, use: python main.py --mode encoder

from encoder_decoder_trainer import EncoderDecoderTranslator, EncoderDecoderTrainer

# Initialize model (without training)
enc_dec_model = EncoderDecoderTranslator(
    model_name="Helsinki-NLP/opus-mt-en-nl",
    learning_rate=2e-5,
    warmup_steps=500
)

print(f"Encoder-Decoder model initialized")
print(f"Total parameters: {sum(p.numel() for p in enc_dec_model.model.parameters()):,}")

## 6. Decoder-Only with LoRA (Quick Demo)

In [ ]:
# This cell demonstrates the LoRA setup
# For full training, use: python main.py --mode decoder

from decoder_only_trainer import DecoderOnlyTranslator

# Initialize model with LoRA (without training)
decoder_model = DecoderOnlyTranslator(
    model_name="bigscience/bloom-560m",
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.1
)

# Print trainable parameters
trainable_params = sum(p.numel() for p in decoder_model.model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in decoder_model.model.parameters())

print(f"Decoder-Only model with LoRA initialized")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

## 7. Results Summary

In [ ]:
# Create results summary DataFrame
results_df = pd.DataFrame({
    'Metric': ['BLEU', 'BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'chrF', 'TER'],
    'Baseline': [
        bleu_result.score,
        bleu_result.precisions[0],
        bleu_result.precisions[1],
        bleu_result.precisions[2],
        bleu_result.precisions[3],
        chrf_result.score,
        ter_result.score
    ]
})

print("\nBaseline Results Summary:")
print(results_df.to_string(index=False))

## 8. Sample Translations Analysis

In [ ]:
# Create detailed translation comparison
comparison_df = pd.DataFrame({
    'Source (English)': sources,
    'Baseline Translation': baseline_translations,
    'Reference (Dutch)': references
})

# Display first 10 samples
comparison_df.head(10)

In [ ]:
# Save results
comparison_df.to_excel('../outputs/evaluation/notebook_baseline_comparison.xlsx', index=False)
print("Results saved to outputs/evaluation/notebook_baseline_comparison.xlsx")

## Next Steps

To run the full training pipeline:

```bash
# From project root
python main.py --mode all
```

This will:
1. Fine-tune the encoder-decoder model (MarianMT)
2. Fine-tune the decoder-only model with LoRA (BLOOM)
3. Evaluate all models on FLORES and software domain test sets
4. Generate comparison reports